# Gear 2.2 · random-anchor persistence baseline (scratch)

Frozen days/anchors under `research/output/gear22_random_anchor_persistence/`.
No canon / Trade_Lat retune.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('research/output/gear22_random_anchor_persistence')
print(json.dumps(json.loads((OUT/'verdict.json').read_text()), indent=2))
print('days', pd.read_csv(OUT/'frozen_days.csv')['day'].tolist())
print('n_anchors', len(pd.read_csv(OUT/'frozen_anchors.csv')))
summary = pd.read_csv(OUT/'latency_summary.csv')
counts = pd.read_csv(OUT/'cell_counts.csv')
delays = pd.read_csv(OUT/'actual_fill_delay.csv')
loo_day = pd.read_csv(OUT/'loo_day.csv')
loo_coin = pd.read_csv(OUT/'loo_coin.csv')
h1 = summary[summary.h_name=='h1'].copy()

In [ ]:
def pool(df):
    rows=[]
    for (W,zb,L),g in df.groupby(['W_ms','z_bin','L_ms']):
        w=g['n_observations'].to_numpy(float)
        if w.sum()<=0: continue
        rows.append({
            'W_ms':W,'z_bin':zb,'L_ms':L,
            'A': np.average(g['asymmetry'], weights=w),
            'p_coll': np.average(g['p_collapse'], weights=w),
            'p_exp': np.average(g['p_expand'], weights=w),
            'p_approx': np.average(g['p_approx'], weights=w),
            'n': int(w.sum()),
        })
    return pd.DataFrame(rows)

P = pool(h1)
zs = ['z_le_1','z_1_2','z_2_4','z_gt_4']
fig, axes = plt.subplots(2,2, figsize=(10,7), sharex=True)
for ax, zb in zip(axes.ravel(), zs):
    for W, ls in [(0,'-'),(20,'--'),(50,'-.'),(100,':')]:
        g = P[(P.z_bin==zb)&(P.W_ms==W)].sort_values('L_ms')
        ax.plot(g.L_ms, g.A, ls, label=f'W={W}')
    ax.axhline(0, color='k', lw=0.5)
    ax.axvspan(40,110, color='0.85', zorder=0)
    ax.set_title(zb); ax.set_xlabel('L ms'); ax.set_ylabel('A(L,1)')
axes[0,0].legend(fontsize=8)
fig.suptitle('Asymmetry A(L,1) by W (candidate CP band annotated only)')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(10,4), sharex=True)
for ax, zb in zip(axes, ['z_2_4','z_gt_4']):
    for W in [0,100]:
        g=P[(P.z_bin==zb)&(P.W_ms==W)].sort_values('L_ms')
        ax.plot(g.L_ms, g.p_coll, label=f'p- W={W}')
        ax.plot(g.L_ms, g.p_exp, '--', label=f'p+ W={W}')
    ax.set_title(zb); ax.legend(fontsize=8); ax.set_xlabel('L ms')
fig.suptitle('p_collapse vs p_expand (h=1)')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7,4))
for zb in zs:
    g=P[(P.z_bin==zb)&(P.W_ms==0)].sort_values('L_ms')
    ax.plot(g.L_ms, g.p_approx, label=zb)
ax.set_title('p_approx(|u|<1), W=0'); ax.legend(); ax.set_xlabel('L ms')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
dmed = delays.groupby('L_ms')['delay_p50'].median()
ax.plot(dmed.index, dmed.values, label='delay p50')
ax.plot(dmed.index, dmed.index, 'k--', lw=0.8, label='delay=L')
ax.set_title('Actual fill delay vs requested L'); ax.legend(); ax.set_xlabel('requested L ms')
plt.show()

heat = counts.pivot_table(index='z_bin', columns='W_ms', values='n_observations_L100', aggfunc='sum')
gate = counts.pivot_table(index='z_bin', columns='W_ms', values='power_gate_pass', aggfunc='max')
fig, ax = plt.subplots(figsize=(6,3))
im = ax.imshow(heat.reindex(zs), aspect='auto')
ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns)
ax.set_yticks(range(len(zs))); ax.set_yticklabels(zs)
ax.set_title('Counts heatmap (L=100 obs sum sides)'); fig.colorbar(im, ax=ax)
plt.show()
print('power gate:\n', gate.reindex(zs))

In [ ]:
h2 = summary[summary.h_name=='h2']
P2 = pool(h2)
fig, ax = plt.subplots(figsize=(7,4))
for zb in ['z_2_4','z_gt_4']:
    g=P2[(P2.z_bin==zb)&(P2.W_ms==0)].sort_values('L_ms')
    ax.plot(g.L_ms, g.A, label=f'{zb} h=2')
ax.axhline(0,color='k',lw=0.5); ax.legend(); ax.set_title('Sensitivity A(L,h=2), W=0')
plt.show()

fig, axes = plt.subplots(1,2, figsize=(10,3.5))
axes[0].hist(loo_day['delta'].dropna(), bins=30); axes[0].set_title('LOO-day ΔA @L=100')
axes[1].hist(loo_coin['delta'].dropna(), bins=30); axes[1].set_title('LOO-coin ΔA @L=100')
plt.tight_layout(); plt.show()